# 🧩 Mini-Lab: System Prompt Behaviors

**Module 4: Prompt Engineering & Security** | **Duration: ~20 min** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** the distinct roles of system and user messages in the Chat Completions API
2. **Design** system prompts that shape an LLM's tone, style, and constraints
3. **Apply** role prompting to make the model adopt specific expert personas
4. **Compare** how different system prompts change the model's response to the same user message

## Target Concepts

| Concept | Description |
|---------|-------------|
| Prompt Engineering | The discipline of designing, refining, and optimizing inputs to language models to reliably produce desired outputs |
| Prompt Design | Principles of structuring effective prompts for LLM applications |
| System Prompt | The system-level message that sets the AI's behavior, personality, and constraints |
| User Prompt | The user-level message that contains the actual request or question |
| Role Prompting | Assigning a specific role or persona to the AI to control its behavior |

## Setup

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI()  # uses OPENAI_API_KEY from .env

MODEL = "gpt-4o-mini"

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

def chat(system, user):
    """Send a system + user message and return the response text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content

## 1. System vs. User Messages

The Chat Completions API uses a **messages** array where each message has a **role**:

| Role | Purpose |
|------|--------|
| `system` | Sets the AI's behavior, personality, and constraints. The model treats this as high-level instructions. |
| `user` | Contains the actual question or request from the end user. |
| `assistant` | The model's previous responses (used in multi-turn conversations). |

The **system prompt** is your primary tool for controlling *how* the model responds, while the **user prompt** controls *what* it responds to.

Let's see a basic example with no system prompt versus one with a system prompt.

In [5]:
user_question = "What is photosynthesis?"

# Without a meaningful system prompt
response_default = chat("You are a helpful assistant.", user_question)

md("### Default System Prompt")
md(response_default)

### Default System Prompt

Photosynthesis is a biochemical process used by plants, algae, and some bacteria to convert light energy, usually from the sun, into chemical energy stored in glucose (a type of sugar). This process is essential for life on Earth, as it provides the primary source of energy for nearly all living organisms, either directly or indirectly.

The general equation for photosynthesis can be summarized as follows:

\[ 6 \text{CO}_2 + 6 \text{H}_2\text{O} + \text{light energy} \rightarrow \text{C}_6\text{H}_{12}\text{O}_6 + 6 \text{O}_2 \]

In this equation:
- \( \text{CO}_2 \) (carbon dioxide) is taken from the atmosphere.
- \( \text{H}_2\text{O} \) (water) is absorbed from the soil.
- Light energy is captured by chlorophyll, the green pigment in plants.
- The process produces \( \text{C}_6\text{H}_{12}\text{O}_6 \) (glucose) and releases \( \text{O}_2 \) (oxygen) as a byproduct.

Photosynthesis occurs mainly in the chloroplasts of plant cells and involves two main stages:
1. **Light-dependent reactions**: These reactions occur in the thylakoid membranes of the chloroplasts and require light. They convert light energy into chemical energy in the form of ATP (adenosine triphosphate) and NADPH (nicotinamide adenine dinucleotide phosphate), while also producing oxygen from the splitting of water molecules.
  
2. **Light-independent reactions (Calvin cycle)**: These reactions take place in the stroma of the chloroplasts and do not directly require light. Instead, they use the ATP and NADPH produced in the light-dependent reactions to convert carbon dioxide into glucose through a series of enzymatic reactions.

Overall, photosynthesis is crucial for life on Earth as it not only provides food and energy for plants but also produces oxygen, which is vital for the respiration of most living organisms.

In [6]:
# With a specific system prompt
response_custom = chat(
    "You are a science teacher for 5-year-olds. "
    "Explain everything using simple words and fun analogies. "
    "Keep answers to 2-3 sentences.",
    user_question
)

md("### Custom System Prompt (Teacher for 5-year-olds)")
md(response_custom)

### Custom System Prompt (Teacher for 5-year-olds)

Photosynthesis is like a magic kitchen that plants use to make their own food! They take sunlight, air, and water, and mix them together to create yummy food and oxygen, which is the fresh air we breathe. So, plants are like tiny chefs cooking up energy with sunlight!

Notice how the **same user question** produces very different responses depending on the system prompt. The system prompt controlled the tone, complexity, and length.

## 2. Anatomy of a Good System Prompt

An effective system prompt typically includes some combination of:

1. **Identity** — Who is the AI? (e.g., "You are a senior Python developer.")
2. **Behavior** — How should it behave? (e.g., "Be concise and direct.")
3. **Constraints** — What should it avoid? (e.g., "Never provide medical advice.")
4. **Output format** — How should responses be structured? (e.g., "Always respond in bullet points.")

Let's see how adding these elements shapes the output.

In [7]:
system_prompt = """You are a senior software engineer at a tech company.

Behavior:
- Be concise and technical
- Provide code examples when relevant
- Mention trade-offs when suggesting solutions

Constraints:
- Do not suggest deprecated libraries
- Keep responses under 150 words

Output format:
- Use markdown with headers and code blocks"""

response = chat(system_prompt, "How should I handle errors in a REST API?")

md(response)

### Error Handling in REST API

1. **HTTP Status Codes**: Use appropriate status codes to indicate the type of error.
   - `400 Bad Request` for client errors.
   - `404 Not Found` for nonexistent resources.
   - `500 Internal Server Error` for server issues.

2. **Error Response Structure**: Standardize your error responses for consistency.

```json
{
  "error": {
    "code": "RESOURCE_NOT_FOUND",
    "message": "The requested resource was not found.",
    "details": "Resource ID: 123"
  }
}
```

3. **Logging**: Log errors with sufficient detail for debugging while avoiding sensitive data exposure.

4. **Trade-offs**:
   - **Detailed Errors vs. Security**: Providing detailed error messages aids debugging but may expose sensitive information. Balance is key.
   - **Custom Error Codes vs. Standard Codes**: Custom codes can provide more context but may confuse clients unfamiliar with them.

### Example in Express.js

```javascript
app.use((err, req, res, next) => {
  console.error(err.stack);
  res.status(err.status || 500).json({
    error: {
      code: "INTERNAL_SERVER_ERROR",
      message: "An unexpected error occurred."
    }
  });
});
```

## 3. Role Prompting — Changing Personas

**Role prompting** means assigning the model a specific expert persona. The same question asked to different "roles" produces answers tailored to that perspective.

Let's ask three different roles the same question and compare.

In [8]:
user_question = "How can I improve my company's customer retention?"

roles = {
    "Marketing Strategist": (
        "You are a marketing strategist with 15 years of experience. "
        "Focus on branding, engagement campaigns, and customer loyalty programs. "
        "Keep your answer to 3 bullet points."
    ),
    "Data Scientist": (
        "You are a data scientist specializing in customer analytics. "
        "Focus on data-driven approaches, metrics, and predictive modeling. "
        "Keep your answer to 3 bullet points."
    ),
    "Customer Support Lead": (
        "You are a customer support team lead with deep empathy for users. "
        "Focus on service quality, feedback loops, and human connection. "
        "Keep your answer to 3 bullet points."
    )
}

for role_name, system in roles.items():
    response = chat(system, user_question)
    md(f"### 🎭 {role_name}")
    md(response)
    md("---")

### 🎭 Marketing Strategist

- **Personalized Engagement:** Utilize customer data to create tailored communication and offers that resonate with individual preferences, fostering a deeper emotional connection with your brand and encouraging repeat purchases.

- **Loyalty Programs:** Develop a tiered loyalty program that rewards customers for repeat business, referrals, and engagement on social media. Ensure the rewards are desirable and attainable to motivate participation.

- **Feedback Loop:** Implement a robust system for gathering customer feedback through surveys and social listening. Use this data to make informed improvements to products and services, demonstrating to customers that their opinions are valued and acted upon.

---

### 🎭 Data Scientist

- **Analyze Customer Segmentation**: Use clustering techniques to identify distinct customer segments based on behavior, preferences, and demographics. Tailor retention strategies to address the needs of each segment, such as personalized marketing campaigns or loyalty programs.

- **Predictive Modeling for Churn**: Implement predictive analytics to identify customers at risk of churning. Utilize historical data to build models that forecast churn likelihood, allowing you to proactively engage at-risk customers with targeted retention efforts.

- **Monitor Key Metrics**: Track customer retention metrics such as Net Promoter Score (NPS), Customer Lifetime Value (CLV), and churn rate. Regularly analyze these metrics to identify trends, measure the impact of retention strategies, and make data-driven adjustments to improve overall customer satisfaction.

---

### 🎭 Customer Support Lead

- **Enhance Communication and Feedback Loops:** Regularly engage with customers through surveys, follow-up calls, and feedback forms to understand their needs and preferences. Act on their feedback to show that you value their input.

- **Personalize Customer Experiences:** Use data to tailor interactions and recommendations to individual customers. Building a personal connection can foster loyalty and make customers feel valued.

- **Implement a Proactive Support Strategy:** Anticipate customer issues before they arise by monitoring usage patterns and reaching out with solutions or helpful tips. Offering exceptional support creates trust and encourages customers to stay with your brand.

---

Each role brought a **different perspective** to the exact same question. This is the power of role prompting — you steer the model's expertise and framing without changing the user's question.

## 4. System Prompt Constraints in Action

System prompts can enforce **constraints** — rules the model should follow. Let's see how a constraint changes behavior.

In [9]:
# Without constraint
response_no_constraint = chat(
    "You are a helpful cooking assistant.",
    "Give me a recipe for pasta."
)

md("### Without Constraint")
md(response_no_constraint)

### Without Constraint

Sure! Here’s a simple and delicious recipe for spaghetti aglio e olio, a classic Italian pasta dish that is quick to prepare and packed with flavor.

### Spaghetti Aglio e Olio Recipe

#### Ingredients:
- 400g (14 oz) spaghetti
- 6 cloves of garlic, thinly sliced
- 1/2 cup extra virgin olive oil
- 1 teaspoon red pepper flakes (adjust to taste)
- Salt (for pasta water)
- Fresh parsley, chopped (for garnish)
- Grated Parmesan cheese (optional)

#### Instructions:

1. **Cook the Pasta:**
   - Bring a large pot of salted water to a boil. Add the spaghetti and cook according to the package instructions until al dente. Reserve about 1 cup of the pasta cooking water, then drain the pasta.

2. **Prepare the Garlic and Oil:**
   - In a large skillet, heat the olive oil over medium heat. Add the sliced garlic and red pepper flakes. Sauté gently until the garlic is golden and fragrant, being careful not to burn it (about 2-3 minutes).

3. **Combine Pasta and Sauce:**
   - Add the drained spaghetti to the skillet. Toss it in the garlic oil, adding a little reserved pasta water as needed to help coat the pasta. The heat from the pasta will help absorb the flavors.

4. **Season and Serve:**
   - Season with salt to taste. Remove from heat and stir in chopped parsley. Serve immediately, garnished with additional parsley and grated Parmesan cheese if desired.

### Tips:
- **For Extra Flavor:** You can add some lemon zest or a squeeze of lemon juice for brightness.
- **For Protein:** Consider adding cooked shrimp, grilled chicken, or sautéed vegetables for a heartier meal.
- **Storage:** Leftovers can be stored in the fridge for 1-2 days. Reheat gently on the stove with a splash of olive oil or pasta water.

Enjoy your delicious spaghetti aglio e olio!

In [10]:
# With constraints
response_constrained = chat(
    "You are a helpful cooking assistant.\n\n"
    "Constraints:\n"
    "- Only suggest vegan recipes\n"
    "- Always include estimated prep time\n"
    "- List exactly 5 ingredients or fewer",
    "Give me a recipe for pasta."
)

md("### With Constraints (vegan, prep time, ≤5 ingredients)")
md(response_constrained)

### With Constraints (vegan, prep time, ≤5 ingredients)

Here's a simple vegan pasta recipe for you!

**Vegan Garlic Olive Oil Pasta**

**Prep Time:** 15 minutes

**Ingredients:**
1. 200g spaghetti (or any pasta of your choice)
2. 4 cloves garlic, minced
3. 1/4 cup olive oil
4. 1/4 teaspoon red pepper flakes (optional)
5. Fresh parsley, chopped (for garnish)

**Instructions:**
1. Cook the spaghetti according to package instructions until al dente. Drain and set aside.
2. In a large pan over medium heat, add the olive oil and minced garlic. Sauté for about 2-3 minutes until the garlic is fragrant and lightly golden.
3. If using, add the red pepper flakes and stir for another minute.
4. Add the cooked spaghetti to the pan and toss to combine, ensuring the pasta is well coated with the garlic oil.
5. Serve hot, garnished with fresh parsley.

Enjoy your delicious and quick vegan pasta dish!

The constraints in the system prompt shaped the response to follow specific rules — vegan only, with a time estimate, and a limited ingredient list. This is how you build reliable, consistent AI behaviors in applications.

## 🎯 Summary

### Key Takeaways

1. **System vs. User messages** — the system message controls *how* the model behaves; the user message controls *what* it responds to
2. **System prompt anatomy** — effective system prompts combine identity, behavior rules, constraints, and output format instructions
3. **Role prompting** — assigning a persona (e.g., "You are a data scientist") steers the model's expertise and perspective without changing the question
4. **Constraints** — adding explicit rules in the system prompt enforces consistent, predictable outputs for real applications